# Descripción y análisis exploratorio de datos

**Materia:** Análisis de Series de Tiempo y Pronósticos (1C-2026)

**Grupo:** 9

**Integrantes:**
- Lucas Achaval - Email: completar
- Marcos Achaval - Email: machavalrodriguez@unsam-bue.edu.ar

**Enlace al conjunto de datos original:** Dataset propio de estación meteorológica de club náutico en Potrerillos (archivo local: data/station_15338.csv).

## Descripción breve del conjunto de datos y su origen

Este conjunto de datos proviene de una estación meteorológica instalada en un club náutico en Potrerillos, Mendoza. La estación registra una observación por minuto con variables de viento (promedio y máximo), dirección, temperatura, humedad relativa y presión atmosférica.

El archivo contiene datos desde 2025-08-01 hasta 2026-04-14, y representa mediciones reales en un entorno de montaña donde el viento térmico tiene un comportamiento diario marcado.

## Problema o pregunta a resolver

Potrerillos, y en particular la zona del embalse, presenta un comportamiento del viento y del clima muy característico. Se trata de un entorno montañoso donde predominan los vientos térmicos, es decir, vientos condicionados por diferencias de temperatura a lo largo del día.

Durante el verano, el régimen de viento suele seguir un patrón muy definido: las intensidades son bajas durante la madrugada, aumentan progresivamente por la mañana, alcanzan valores más estables al mediodía y luego comienzan a disminuir por la tarde y la noche. La dirección del viento también responde a una dinámica clara en esta época del año. En general, sopla desde el sector sudeste (135°).

Sin embargo, a pesar de este patrón general, los pronósticos de viento para esta zona suelen fallar en la representación de la intensidad observada. Con frecuencia subestiman los máximos diarios y no capturan con suficiente precisión la señal térmica local. Por eso, nuestro objetivo con este proyecto es desarrollar un modelo de pronóstico de viento para este lugar que sea más preciso que los existentes y que capture adecuadamente los efectos térmicos. Para ello nos vamos a enfocar específicamente en el pronóstico de la intensidad del viento (variable objetivo: `wind_avg`), y no en su dirección.

Métricas de evaluación propuestas:
- MAE (error absoluto medio)
- RMSE (raíz del error cuadrático medio)
- R2

El horizonte de pronóstico será de 12 horas y se generarán 12 predicciones a futuro, una por cada hora. Para ello, primero se realizará una agregación temporal (resample) del conjunto de datos a resolución horaria mediante la mediana, con el fin de alinear la escala temporal del modelo con el objetivo del pronóstico.

In [6]:
import pandas

df = pandas.read_csv("../data/station_15338.csv")
df.set_index("datetime", inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 350492 entries, 2025-08-01 00:01:00 to 2026-04-14 00:00:00
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   unixtime        350492 non-null  int64  
 1   wind_avg        350492 non-null  float64
 2   wind_max        350492 non-null  float64
 3   wind_min        0 non-null       float64
 4   wind_direction  350491 non-null  float64
 5   temperature     350492 non-null  float64
 6   rh              350492 non-null  float64
 7   mslp            350473 non-null  float64
 8   gustiness       346917 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 26.7+ MB


In [5]:
df.describe()

,unixtime,wind_avg,wind_max,wind_min,wind_direction,temperature,rh,mslp,gustiness
count,3.504920e+05,350492.000000,350492.000000,0.0,350491.000000,350492.000000,350492.000000,350473.000000,346917.000000
mean,1.765055e+09,5.468075,8.145345,NaN,118.535537,15.950020,54.729366,861.521302,72.649922
std,6.426977e+06,4.124164,5.472402,NaN,63.922672,6.513114,18.586506,4.037689,93.689089
min,1.754017e+09,0.000000,0.000000,NaN,0.000000,-3.200000,4.800000,847.000000,0.000000
25%,1.759472e+09,2.200000,3.700000,NaN,70.000000,11.500000,40.300000,859.000000,31.000000
50%,1.764963e+09,3.800000,5.800000,NaN,108.000000,16.400000,54.300000,861.400000,47.000000
75%,1.770780e+09,8.500000,12.600000,NaN,154.000000,20.800000,69.000000,864.000000,79.000000
max,1.776136e+09,46.000000,74.500000,NaN,360.000000,32.500000,99.000000,873.600000,3234.000000
